# V24: 8B QLoRA feasibility only

No competition predictions or accuracy claims. Method: Dettmers et al., QLoRA (arXiv:2305.14314); implementation uses current Hugging Face PEFT and bitsandbytes.

In [ ]:
import subprocess,sys,importlib.metadata
from packaging.version import Version
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)

import glob,hashlib
from pathlib import Path
wheels=glob.glob('/kaggle/input/**/bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl',recursive=True)
assert len(wheels)==1,wheels
assert hashlib.sha256(Path(wheels[0]).read_bytes()).hexdigest()=='173d137610468bec9cddbaa2e049254e97792657ab984e3e737bec1772c1668c'
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall',wheels[0]],check=True)
assert importlib.metadata.version('bitsandbytes')=='0.50.0'


In [ ]:
import json,os,subprocess,sys
from pathlib import Path
Path('plan.json').write_text('{"experiment": "improvement-v24", "kind": "8B QLoRA bounded feasibility probe", "seed": 2401, "microsteps": 24, "wall_limit_seconds": 1200, "model": "qwen-lm/qwen-3/Transformers/8b-base/1", "train_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707", "worker_sha256": "0c6d158e53ea8f87f09d1dd7228591f890b582de5e3bdb4cc0b96eaf5aec001b", "quantization": "NF4 double quant, fp16 compute", "adapter": "all-linear rank16 alpha32", "training": "CE only, batch4 accumulation8, max256, AdamW 1e-4, longest32 host passages", "purpose": "Measure finite updates, memory, throughput and adapter reload before any full pilot", "qualification": "Completed finite updates and reload; projected training plus measured overhead must fit 9h with 25% margin before a full pilot", "sources": ["https://arxiv.org/abs/2305.14314", "https://github.com/artidoro/qlora", "https://huggingface.co/docs/peft/developer_guides/quantization", "https://huggingface.co/docs/transformers/quantization/bitsandbytes"], "source_assessment": "Authors paper/repository establish method, current Hugging Face docs establish API. bitsandbytes official repository not archived, pushed 2026-09-07 as observed 2026-10-26. Historical QLoRA claims are not competition accuracy evidence.", "changes_from_paper": "Uses fp16 compute on T4 and ordinary AdamW; no paged optimizer, not instruction tuning. Probe is not an accuracy experiment.", "setup_correction": "V5 explicitly enters train mode and asserts checkpointing active; V4 OOM before first microstep, not a valid checkpointed memory measurement"}')
Path('probe.py').write_text('"""Bounded 8B NF4 feasibility probe; does not estimate competition accuracy."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nos.environ[\'HF_HUB_OFFLINE\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom safetensors.torch import load_file\nfrom peft import set_peft_model_state_dict\n\n\ndef main():\n    started = time.monotonic()\n    plan = json.loads(Path(\'plan.json\').read_text())\n    def log(**values):\n        record = dict(seconds=time.monotonic()-started, **values)\n        with Path(\'progress.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record), flush=True)\n    torch.manual_seed(plan[\'seed\'])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    configs = []\n    for path in Path(\'/kaggle/input\').rglob(\'config.json\'):\n        cfg = json.loads(path.read_text())\n        log(event=\'input_config\', path=str(path), model_type=cfg.get(\'model_type\'),\n            hidden_size=cfg.get(\'hidden_size\'), num_hidden_layers=cfg.get(\'num_hidden_layers\'))\n        if cfg.get(\'model_type\') == \'qwen3\' and cfg.get(\'hidden_size\') == 4096 and cfg.get(\'num_hidden_layers\') == 36:\n            configs.append(path.parent)\n    assert len(configs) == 1, configs\n    model_dir = configs[0]\n    train_files = list(Path(\'/kaggle/input\').rglob(\'train.csv\'))\n    assert len(train_files) == 1\n    assert hashlib.sha256(train_files[0].read_bytes()).hexdigest() == plan[\'train_sha256\']\n    train = pd.read_csv(train_files[0])\n    labels = sorted(train.model.unique())\n    # Long real passages exercise the full padded sequence length, with no validation selection.\n    frame = train.assign(length=train.text.str.len()).sort_values([\'length\', \'id\'], ascending=[False, True]).head(32)\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = \'right\'\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tokens = tok(frame.text.tolist(), add_special_tokens=False, truncation=True, max_length=255)[\'input_ids\']\n    tokens = [ids+[tok.eos_token_id] for ids in tokens]\n    bank = tok.pad({\'input_ids\': tokens}, padding=\'max_length\', max_length=256, return_tensors=\'pt\')\n    targets = torch.tensor([labels.index(label) for label in frame.model], device=\'cuda:0\')\n    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n    backbone = AutoModel.from_pretrained(model_dir, quantization_config=quant, torch_dtype=torch.float16,\n        device_map={\'\': 0}, local_files_only=True, attn_implementation=\'sdpa\')\n    backbone.config.use_cache = False\n    backbone = prepare_model_for_kbit_training(backbone,\n        gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    encoder = get_peft_model(backbone, LoraConfig(r=16, lora_alpha=32,\n        target_modules=\'all-linear\', lora_dropout=0.0, bias=\'none\'))\n    head = nn.Sequential(nn.LayerNorm(4096), nn.Linear(4096, 18)).cuda()\n    trainable = [p for p in encoder.parameters() if p.requires_grad]+list(head.parameters())\n    for parameter in trainable:\n        parameter.data = parameter.data.float()\n    assert all(\'lora_\' in name for name,p in encoder.named_parameters() if p.requires_grad)\n    initial = {name: p.detach().cpu().clone() for name,p in encoder.named_parameters() if p.requires_grad}\n    optimizer = torch.optim.AdamW(trainable, lr=1e-4, weight_decay=0.01)\n    scaler = torch.amp.GradScaler(\'cuda\', init_scale=1024)\n    log(event=\'loaded\', versions={name:importlib.metadata.version(name) for name in\n        [\'torch\',\'transformers\',\'peft\',\'bitsandbytes\',\'accelerate\']}, gpu=torch.cuda.get_device_name(0),\n        trainable_parameters=sum(p.numel() for p in trainable), model_config_sha256=hashlib.sha256((model_dir/\'config.json\').read_bytes()).hexdigest())\n    def forward(batch):\n        hidden = encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch[\'attention_mask\'].clone()\n        mask[:, 0] = 0\n        pooled = (hidden*mask.unsqueeze(-1)).sum(1)/mask.sum(1, keepdim=True).clamp_min(1)\n        return head(pooled)\n    elapsed = []\n    skipped = 0\n    encoder.train(); head.train()\n    checkpointed = [module for module in encoder.modules() if getattr(module, \'gradient_checkpointing\', False)]\n    assert checkpointed and all(module.training for module in checkpointed)\n    log(event=\'training_mode\', encoder_training=encoder.training,\n        checkpointed_training_modules=len(checkpointed))\n    optimizer.zero_grad(set_to_none=True)\n    torch.cuda.reset_peak_memory_stats()\n    for step in range(plan[\'microsteps\']):\n        assert time.monotonic()-started < plan[\'wall_limit_seconds\']\n        ids = np.arange((step%8)*4, (step%8+1)*4)\n        batch = {key:value[ids].cuda() for key,value in bank.items()}\n        torch.cuda.synchronize()\n        tick = time.monotonic()\n        with torch.autocast(\'cuda\', dtype=torch.float16):\n            logits = forward(batch)\n            loss = nn.functional.cross_entropy(logits.float(), targets[ids])\n        assert torch.isfinite(loss), \'nonfinite loss\'\n        scaler.scale(loss/8).backward()\n        if (step+1)%8 == 0:\n            scaler.unscale_(optimizer)\n            norm = torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n            before = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            skipped += int(scaler.get_scale() < before)\n            log(event=\'update\', step=step, grad_norm=float(norm), skipped=skipped)\n            optimizer.zero_grad(set_to_none=True)\n        torch.cuda.synchronize()\n        elapsed.append(time.monotonic()-tick)\n        log(event=\'step\', step=step, loss=float(loss.detach()), step_seconds=elapsed[-1],\n            peak_allocated=torch.cuda.max_memory_allocated(), peak_reserved=torch.cuda.max_memory_reserved())\n    changed = sum(not torch.equal(initial[name],p.detach().cpu()) for name,p in encoder.named_parameters() if p.requires_grad)\n    assert changed > 0 and skipped < plan[\'microsteps\']//8\n    assert all(torch.isfinite(p).all() for p in trainable)\n    encoder.save_pretrained(\'adapter\', safe_serialization=True)\n    head.eval(); encoder.eval()\n    with torch.no_grad(), torch.autocast(\'cuda\', dtype=torch.float16):\n        before = forward(batch).detach().float().cpu()\n    # Perturb and restore adapters to exercise actual saved tensor loading.\n    for p in encoder.parameters():\n        if p.requires_grad:\n            p.data.zero_()\n    state = load_file(\'adapter/adapter_model.safetensors\')\n    loaded = set_peft_model_state_dict(encoder, state)\n    assert not loaded.unexpected_keys\n    assert not [key for key in loaded.missing_keys if \'lora_\' in key]\n    with torch.no_grad(), torch.autocast(\'cuda\', dtype=torch.float16):\n        after = forward(batch).detach().float().cpu()\n    torch.testing.assert_close(before, after, rtol=1e-5, atol=1e-5)\n    measured = elapsed[8:]\n    result = dict(completed=True, kind=\'feasibility only, no accuracy result\', microsteps=len(elapsed),\n        changed_tensors=changed, skipped_updates=skipped, reload_check=\'passed\',\n        median_microstep_seconds=float(np.median(measured)), max_microstep_seconds=max(measured),\n        projected_5580_microsteps_seconds=float(np.mean(measured)*5580),\n        peak_allocated_bytes=torch.cuda.max_memory_allocated(), peak_reserved_bytes=torch.cuda.max_memory_reserved(),\n        elapsed_seconds=time.monotonic()-started,\n        caveats=[\'CE only, not SupCon; fixed length 256; throughput projection excludes loading and readout\',\n                 \'No validation or public score; all-linear adapters differ from earlier q/v-only experiments\'])\n    Path(\'result.json\').write_text(json.dumps(result, indent=2))\n    log(event=\'complete\', **result)\n\n\nif __name__ == \'__main__\':\n    main()\n')
subprocess.run([sys.executable,'-u','probe.py'],env=dict(os.environ,CUDA_VISIBLE_DEVICES='0'),check=True,timeout=1200)
